In [55]:
import os
import torch
from torch.nn.functional import cross_entropy
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_from_disk, load_dataset
from tqdm import tqdm

In [2]:
main_device = 'auto'
os.environ["CUDA_VISIBLE_DEVICES"] = "6,7"

In [3]:
torch.random.manual_seed(19260817)
TOTAL_ENTRIES = 1

In [4]:
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-30B-A3B", device_map=main_device, torch_dtype=torch.bfloat16, attn_implementation="flash_attention_2",)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-30B-A3B", device_map=main_device)

dataset = load_dataset("lmsys/lmsys-chat-1m", split="train")

Loading checkpoint shards: 100%|██████████| 16/16 [00:28<00:00,  1.77s/it]


In [5]:
model.eval()
model.config.use_cache = False
model.config.output_logits = True

In [6]:
def calc_perplexity(logits, token_ids):
  assert logits.shape[:-1] == token_ids.shape, \
      f"Logits shape {logits.shape} does not match token_ids shape {token_ids.shape}"
  loss = cross_entropy(logits.view(-1, logits.size(-1)), token_ids.view(-1), reduction='none')
  perplexity = torch.exp(loss)
  return perplexity.view(token_ids.shape)


In [8]:
# for entry in tqdm(dataset.take(TOTAL_ENTRIES).to_iterable_dataset(), total=TOTAL_ENTRIES):
for entry in tqdm(dataset.select(range(TOTAL_ENTRIES))):
  chat = tokenizer.apply_chat_template(entry["conversation"], tokenize=False)
  inputs = tokenizer(chat, return_tensors="pt").to(model.device)

  with torch.no_grad():
    outputs = model(**inputs)

  input_ids = inputs.input_ids[0]
  input_length = input_ids.size(0)
  target_ids = input_ids[1:]

  # Calculate perplexity from logits.
  # Remove the last token to avoid sampling the next token
  output_logits = outputs.logits[0, :-1, :]  # [batch_size, seq_len, vocab_size]
  full_perplexity = calc_perplexity(output_logits, target_ids)
  print(f"Full Perplexity: {full_perplexity.mean().item()}")

  base_top_k = model.config.num_experts_per_tok
  token_top_ks = torch.where(full_perplexity > 1.2, base_top_k, base_top_k-1)
  # Append the last token to match the input length
  token_top_ks = torch.cat([token_top_ks, torch.tensor([base_top_k], device=model.device)], dim=0)

  with torch.no_grad():
    outputs = model(
      **inputs,
      token_top_ks=token_top_ks,
    )

  output_logits = outputs.logits[0, :-1, :]
  reduced_perplexity = calc_perplexity(output_logits, target_ids)
  print(f"Reduced Perplexity: {reduced_perplexity.mean().item()}")

  0%|          | 0/1 [00:00<?, ?it/s]

Full Perplexity: 113816633344.0
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z toke

100%|██████████| 1/1 [00:02<00:00,  2.78s/it]

Reduced Perplexity: 129385889792.0


In [ ]:
chat = tokenizer.apply_chat_template([
  {
    "role": "user",
    "content": "What is the capital of France?"
  }
], tokenize=False, enable_thinking=True)
inputs = tokenizer(chat, return_tensors="pt").to(model.device)
outputs = model.generate(
  **inputs,
  max_new_tokens=4096,
  return_dict_in_generate=True,
  output_logits=True,
)

In [27]:
temp_conversation = tokenizer.decode(outputs.sequences[0])

In [52]:
inputs = tokenizer(temp_conversation, return_tensors="pt").to(model.device)

with torch.no_grad():
  outputs = model(**inputs)

input_ids = inputs.input_ids[0]
input_length = input_ids.size(0)
target_ids = input_ids[1:]

# Calculate perplexity from logits.
# Remove the last token to avoid sampling the next token
output_logits = outputs.logits[0, :-1, :]  # [batch_size, seq_len, vocab_size]
full_perplexity = calc_perplexity(output_logits, target_ids)
print(f"Full Perplexity: {full_perplexity.mean().item()}")

base_top_k = model.config.num_experts_per_tok
token_top_ks = torch.where(full_perplexity > 2.0, base_top_k, base_top_k-1)
# Append the last token to match the input length
token_top_ks = torch.cat([token_top_ks, torch.tensor([base_top_k], device=model.device)], dim=0)

with torch.no_grad():
  outputs = model(
    **inputs,
    token_top_ks=token_top_ks,
  )

output_logits = outputs.logits[0, :-1, :]
reduced_perplexity = calc_perplexity(output_logits, target_ids)
print(f"Reduced Perplexity: {reduced_perplexity.mean().item()}")

Full Perplexity: 3024.0
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks is applied
Z token_top_ks

In [34]:
full_perplexity

tensor([1.5168e+04, 1.4800e+02, 5.6250e+01, 1.2031e+00, 1.0547e+00, 9.1520e+03,
        1.1094e+00, 1.2562e+01, 5.1562e+00, 1.5401e+06, 1.0000e+00, 1.0938e+00,
        1.0547e+00, 3.1719e+00, 1.0000e+00, 1.0000e+00, 1.3672e+00, 1.3672e+00,
        1.0859e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00,
        1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00,
        1.4766e+00, 1.0000e+00, 1.0000e+00, 1.0859e+00, 1.0469e+00, 1.0000e+00,
        1.0078e+00, 1.0000e+00, 1.0625e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00,
        1.0625e+00, 1.0000e+00, 1.0000e+00, 1.0156e+00, 1.4922e+00, 1.0234e+00,
        1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00,
        1.1719e+00, 1.0000e+00, 1.3828e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00,
        1.0000e+00, 1.3438e+00, 1.0000e+00, 1.1562e+00, 1.0078e+00, 1.0000e+00,
        1.0000e+00, 1.7734e+00, 1.9297e+00, 1.2812e+00, 1.0156e+00, 1.0625e+00,
        1.0078e+00, 1.0000e+00, 1.0000e+

In [42]:
from IPython.display import HTML, display

def get_ppl_html(tokens, ppl_values):
    """
    根据PPL值为token生成HTML字符串，用于可视化。

    这个函数不直接显示任何内容，而是返回一个完整的HTML字符串。

    参数:
    - tokens (list[str]): token的文本列表。
    - ppl_values (list[float]): 与token对应的PPL值列表。

    返回:
    - str: 一个包含可视化内容的完整HTML字符串。
    """
    if not tokens or not ppl_values or len(tokens) != len(ppl_values):
        raise ValueError("输入列表不能为空，且tokens和ppl_values的长度必须一致。")

    # 1. 找到PPL的最大值和最小值
    min_ppl = min(ppl_values)
    max_ppl = max(ppl_values)
    
    # 处理所有PPL值都相同的情况，避免除以零
    if max_ppl == min_ppl:
        # 如果所有值都一样，给一个中间态的固定透明度
        normalized_ppl_values = [0.5] * len(ppl_values)
    else:
        # 2. PPL值归一化到 0-1 范围
        normalized_ppl_values = [(ppl - min_ppl) / (max_ppl - min_ppl) for ppl in ppl_values]

    # --- HTML和CSS内容构建 ---
    
    # 核心HTML片段，不包含<html>, <body>等标签，方便嵌入
    html_body_content = ""
    
    # 3. 为每个token生成带样式的<span>
    for token, normalized_ppl, original_ppl in zip(tokens, normalized_ppl_values, ppl_values):
        # 基础透明度设为0.1，确保最低PPL也有浅色背景
        # PPL越高，alpha越接近1.0
        alpha = 0.1 + 0.9 * normalized_ppl
        
        # 定义颜色
        blue_color = "30, 144, 255" # RGB for Dodger Blue
        
        # HTML编码token以防特殊字符（如 < >）破坏HTML结构
        safe_token = token.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

        # 为每个token创建一个span，并添加一个tooltip来显示精确的PPL值
        span = (
            f'<div class="tooltip">'
            f'<span class="token" style="background-color: rgba({blue_color}, {alpha:.2f});">{safe_token}</span>'
            f'<span class="tooltiptext">PPL: {original_ppl:.2f}</span>'
            f'</div>'
        )
        html_body_content += span + " " # 添加一个空格来分隔token

    # 构建完整的HTML文档（包含CSS样式）
    full_html = f"""
    <style>
        .ppl-vis-container {{ 
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif;
            padding: 10px; 
            font-size: 1.1em; 
            line-height: 2.2;
            border: 1px solid #e0e0e0;
            border-radius: 8px;
        }}
        .token {{
            display: inline-block;
            padding: 3px 5px;
            border-radius: 4px;
            margin: 0 1px;
            transition: all 0.2s ease-in-out;
        }}
        .tooltip {{
            position: relative;
            display: inline-block;
            cursor: default;
        }}
        .tooltip .tooltiptext {{
            visibility: hidden;
            width: 120px;
            background-color: #333;
            color: #fff;
            text-align: center;
            border-radius: 6px;
            padding: 5px 0;
            position: absolute;
            z-index: 1;
            bottom: 130%;
            left: 50%;
            margin-left: -60px;
            opacity: 0;
            transition: opacity 0.3s;
            font-size: 0.8em;
            line-height: 1.4;
        }}
        .tooltip:hover .tooltiptext {{
            visibility: visible;
            opacity: 1;
        }}
        .legend {{
            margin-top: 20px;
            padding-top: 15px;
            border-top: 1px solid #eee;
        }}
        .legend-title {{
            font-weight: bold;
            margin-bottom: 8px;
            font-size: 0.9em;
        }}
        .gradient-bar {{
            height: 18px;
            width: 100%;
            background: linear-gradient(to right, rgba(30, 144, 255, 0.1), rgba(30, 144, 255, 1.0));
            border-radius: 3px;
        }}
        .legend-labels {{
            display: flex;
            justify-content: space-between;
            font-size: 0.8em;
            color: #555;
            margin-top: 4px;
        }}
    </style>
    <div class="ppl-vis-container">
        <div>{html_body_content}</div>
        <div class="legend">
            <div class="legend-title">Perplexity (PPL) Scale</div>
            <div class="gradient-bar"></div>
            <div class="legend-labels">
                <span>Low PPL ({min_ppl:.2f})</span>
                <span>High PPL ({max_ppl:.2f})</span>
            </div>
        </div>
    </div>
    """
    return full_html

def display_ppl_in_notebook(tokens, ppl_values):
    """
    生成PPL可视化并在Jupyter/IPython环境中直接显示。

    参数:
    - tokens (list[str]): token的文本列表。
    - ppl_values (list[float]): 与token对应的PPL值列表。

    返回:
    - IPython.display.HTML: 一个可以被Notebook渲染的HTML对象。
    """
    html_content = get_ppl_html(tokens, ppl_values)
    return HTML(html_content)

In [47]:
tokenizer.batch_decode(target_ids)

['user',
 '\n',
 'What',
 ' is',
 ' the',
 ' capital',
 ' of',
 ' France',
 '?',
 '<|im_end|>',
 '\n',
 '<|im_start|>',
 ':',
 ' \n\n',
 'Okay',
 ',',
 ' so',
 ' the',
 ' user',
 ' is',
 ' asking',
 ',',
 ' "',
 'What',
 ' is',
 ' the',
 ' capital',
 ' of',
 ' France',
 '?"',
 ' Let',
 ' me',
 ' think',
 ' about',
 ' how',
 ' to',
 ' approach',
 ' this',
 '.',
 ' First',
 ',',
 ' I',
 ' need',
 ' to',
 ' recall',
 ' basic',
 ' geographical',
 ' information',
 '.',
 ' France',
 ' is',
 ' a',
 ' country',
 ' in',
 ' Europe',
 ',',
 ' right',
 '?',
 ' I',
 ' remember',
 ' that',
 ' the',
 ' capital',
 ' cities',
 ' of',
 ' countries',
 ' are',
 ' usually',
 ' significant',
 ' cities',
 ',',
 ' sometimes',
 ' the',
 ' largest',
 ' or',
 ' the',
 ' historical',
 ' center',
 '.\n\n',
 'I',
 ' think',
 ' the',
 ' capital',
 ' of',
 ' France',
 ' is',
 ' Paris',
 '.',
 ' Wait',
 ',',
 ' is',
 ' that',
 ' correct',
 '?',
 ' Let',
 ' me',
 ' verify',
 '.',
 ' Paris',
 ' is',
 ' a',
 ' major',
 '

In [53]:
display_ppl_in_notebook(tokenizer.batch_decode(target_ids)[10:], full_perplexity.tolist()[10:])

In [54]:
display_ppl_in_notebook(tokenizer.batch_decode(target_ids)[10:], reduced_perplexity.tolist()[10:])